# 🌱 Plant Disease Prediction Using CNN

Beginner-level tomato leaf classification project.


In [ ]:
from pathlib import Path
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

DATA_DIR = Path("../data/processed")
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 10
CLASS_NAMES = ["early_blight", "healthy", "late_blight"]


## 1. Load the dataset


In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
    class_names=CLASS_NAMES
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    class_names=CLASS_NAMES
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    class_names=CLASS_NAMES
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


## 2. Build a simple CNN


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 128, 3)),
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Conv2D(16, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 3. Train the model


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)


## 4. Plot accuracy and loss


In [ ]:
plt.figure(figsize=(7,5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(7,5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


## 5. Evaluate on test data


In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print("Test Accuracy:", test_accuracy)


## 6. Save the model


In [ ]:
Path("../models").mkdir(exist_ok=True)
model.save("../models/plant_disease_model.keras")
print("Model saved!")


## 7. Make one prediction


In [ ]:
from PIL import Image

image_path = "../sample_images/leaf.jpg"  # replace with your image
image = Image.open(image_path).convert("RGB").resize(IMG_SIZE)
arr = np.array(image, dtype=np.float32)[None, ...]

probabilities = model.predict(arr, verbose=0)[0]
index = np.argmax(probabilities)

print("Prediction:", CLASS_NAMES[index])
print("Confidence:", probabilities[index] * 100)
